# 🇧🇷 Brazilian Health Data with PySUS

This notebook demonstrates how to access Brazilian public health data using the PySUS library.

**Data Sources:**
- **SINAN** - Notifiable Diseases (dengue, malaria, COVID-19, etc.)
- **SIM** - Mortality Information System
- **SIH** - Hospital Information System
- **SIA** - Outpatient Information System
- **CNES** - National Health Establishments Registry

**Requirements:**
```bash
pip install pysus pandas matplotlib seaborn plotly
```

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# PySUS imports
from pysus.online_data import SINAN, SIM, SIH, SIA, CNES
from pysus.preprocessing.decoders import translate_variables

print("✅ Imports completed successfully!")
print(f"⏰ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

## 2. Exploring Available Data

In [ ]:
# List available diseases in SINAN
print("📋 Available diseases in SINAN:")
diseases = SINAN.list_diseases()
for i, disease in enumerate(diseases[:15], 1):
    print(f"  {i:2d}. {disease}")
print(f"  ... and {len(diseases) - 15} more diseases")

## 3. Dengue Data Analysis

In [ ]:
# Download dengue data for Rio de Janeiro (2022-2023)
print("📥 Downloading dengue data for Rio de Janeiro...")

try:
    # Get dengue notifications for RJ in 2023
    dengue_rj_2023 = SINAN.download(year=2023, disease="Dengue", states="RJ")
    
    print(f"✅ Downloaded {len(dengue_rj_2023)} records")
    print(f"\n📊 DataFrame shape: {dengue_rj_2023.shape}")
    print(f"\n📝 Columns: {list(dengue_rj_2023.columns)[:10]}...")
    
    # Display first few rows
    dengue_rj_2023.head()
    
except Exception as e:
    print(f"⚠️ Error downloading data: {e}")
    print("\n💡 Tip: Make sure you have internet connection and PySUS is properly installed")

In [ ]:
# Data exploration
if 'dengue_rj_2023' in locals() and len(dengue_rj_2023) > 0:
    print("🔍 Data Exploration:\n")
    
    # Basic statistics
    print("📈 Basic Statistics:")
    print(f"  Total cases: {len(dengue_rj_2023):,}")
    
    # Age distribution
    if 'NU_IDADE_N' in dengue_rj_2023.columns:
        print(f"\n  Age range: {dengue_rj_2023['NU_IDADE_N'].min():.0f} - {dengue_rj_2023['NU_IDADE_N'].max():.0f} years")
        print(f"  Mean age: {dengue_rj_2023['NU_IDADE_N'].mean():.1f} years")
    
    # Gender distribution
    if 'CS_SEXO' in dengue_rj_2023.columns:
        gender_counts = dengue_rj_2023['CS_SEXO'].value_counts()
        print(f"\n  Gender distribution:")
        for gender, count in gender_counts.items():
            percentage = (count / len(dengue_rj_2023)) * 100
            print(f"    {gender}: {count:,} ({percentage:.1f}%)")
    
    # Evolution status
    if 'EVOLUCAO' in dengue_rj_2023.columns:
        print(f"\n  Evolution status:")
        evol_counts = dengue_rj_2023['EVOLUCAO'].value_counts()
        for status, count in evol_counts.head(5).items():
            print(f"    {status}: {count:,}")

## 4. Visualizations

In [ ]:
# Age distribution histogram
if 'dengue_rj_2023' in locals() and 'NU_IDADE_N' in dengue_rj_2023.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Age histogram
    axes[0].hist(dengue_rj_2023['NU_IDADE_N'].dropna(), bins=30, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Age (years)')
    axes[0].set_ylabel('Number of Cases')
    axes[0].set_title('Dengue Cases by Age - Rio de Janeiro 2023')
    axes[0].grid(True, alpha=0.3)
    
    # Gender distribution
    if 'CS_SEXO' in dengue_rj_2023.columns:
        gender_data = dengue_rj_2023['CS_SEXO'].value_counts()
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
        axes[1].pie(gender_data.values, labels=gender_data.index, autopct='%1.1f%%', 
                   colors=colors, startangle=90)
        axes[1].set_title('Dengue Cases by Gender - Rio de Janeiro 2023')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Time series analysis (if date column available)
if 'dengue_rj_2023' in locals() and 'DT_NOTIFIC' in dengue_rj_2023.columns:
    # Convert date column
    dengue_rj_2023['DT_NOTIFIC'] = pd.to_datetime(dengue_rj_2023['DT_NOTIFIC'], errors='coerce')
    
    # Group by month
    monthly_cases = dengue_rj_2023.groupby(dengue_rj_2023['DT_NOTIFIC'].dt.to_period('M')).size()
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    monthly_cases.plot(kind='bar', ax=ax, color='coral', edgecolor='black')
    ax.set_xlabel('Month')
    ax.set_ylabel('Number of Cases')
    ax.set_title('Dengue Cases by Month - Rio de Janeiro 2023')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Peak month: {monthly_cases.idxmax()} with {monthly_cases.max()} cases")

## 5. Mortality Data (SIM)

In [ ]:
# Download mortality data for São Paulo
print("📥 Downloading mortality data for São Paulo (2022)...")

try:
    # Get mortality data for SP in 2022
    sim_sp_2022 = SIM.download(year=2022, states="SP")
    
    print(f"✅ Downloaded {len(sim_sp_2022):,} death records")
    
    # Top causes of death
    if 'CAUSABAS' in sim_sp_2022.columns:
        print("\n💀 Top 10 Causes of Death (ICD-10 codes):")
        top_causes = sim_sp_2022['CAUSABAS'].value_counts().head(10)
        for i, (cause, count) in enumerate(top_causes.items(), 1):
            print(f"  {i:2d}. {cause}: {count:,} cases")
    
except Exception as e:
    print(f"⚠️ Error downloading SIM data: {e}")

In [ ]:
# Visualize top causes of death
if 'sim_sp_2022' in locals() and 'CAUSABAS' in sim_sp_2022.columns:
    top_10_causes = sim_sp_2022['CAUSABAS'].value_counts().head(10)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    top_10_causes.plot(kind='barh', ax=ax, color='lightcoral', edgecolor='black')
    ax.set_xlabel('Number of Deaths')
    ax.set_ylabel('ICD-10 Cause Code')
    ax.set_title('Top 10 Causes of Death - São Paulo 2022')
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()

## 6. Hospitalizations (SIH)

In [ ]:
# Download hospitalization data
print("📥 Downloading hospitalization data for Minas Gerais (2023)...")

try:
    # Get hospitalization data for MG in 2023
    sih_mg_2023 = SIH.download(year=2023, month=1, states="MG", group="RD")  # RD = AIH Reduced
    
    print(f"✅ Downloaded {len(sih_mg_2023):,} hospitalization records")
    
    if 'VAL_TOT' in sih_mg_2023.columns:
        total_cost = sih_mg_2023['VAL_TOT'].sum()
        mean_cost = sih_mg_2023['VAL_TOT'].mean()
        print(f"\n💰 Hospitalization Costs:")
        print(f"  Total: R$ {total_cost:,.2f}")
        print(f"  Average per hospitalization: R$ {mean_cost:,.2f}")
    
except Exception as e:
    print(f"⚠️ Error downloading SIH data: {e}")

## 7. Comparative Analysis: Multiple States

In [ ]:
# Compare dengue across multiple states
states = ["RJ", "SP", "MG", "BA"]
state_names = {"RJ": "Rio de Janeiro", "SP": "São Paulo", 
               "MG": "Minas Gerais", "BA": "Bahia"}

state_data = {}

print("📥 Downloading dengue data for multiple states...")
for state in states:
    try:
        df = SINAN.download(year=2023, disease="Dengue", states=state)
        state_data[state] = len(df)
        print(f"  {state_names[state]} ({state}): {len(df):,} cases")
    except Exception as e:
        print(f"  ⚠️ {state}: {e}")
        state_data[state] = 0

In [ ]:
# Visualize comparison
if state_data:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    states_list = list(state_data.keys())
    cases_list = list(state_data.values())
    names = [state_names[s] for s in states_list]
    
    bars = ax.bar(names, cases_list, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'], 
                  edgecolor='black')
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height):,}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax.set_ylabel('Number of Dengue Cases')
    ax.set_title('Dengue Cases by State - Brazil 2023')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    total_cases = sum(cases_list)
    print(f"\n📊 Summary:")
    print(f"  Total cases: {total_cases:,}")
    print(f"  Highest: {max(state_data.items(), key=lambda x: x[1])}")
    print(f"  Lowest: {min(state_data.items(), key=lambda x: x[1])}")

## 8. Export Data

In [ ]:
# Save processed data
output_dir = "./output"
import os
os.makedirs(output_dir, exist_ok=True)

if 'dengue_rj_2023' in locals():
    output_file = os.path.join(output_dir, "dengue_rj_2023.csv")
    dengue_rj_2023.to_csv(output_file, index=False)
    print(f"✅ Saved dengue data to: {output_file}")
    print(f"   File size: {os.path.getsize(output_file) / 1024:.1f} KB")

if 'sim_sp_2022' in locals():
    output_file = os.path.join(output_dir, "sim_sp_2022.csv")
    sim_sp_2022.to_csv(output_file, index=False)
    print(f"✅ Saved mortality data to: {output_file}")
    print(f"   File size: {os.path.getsize(output_file) / 1024:.1f} KB")

print(f"\n📁 All outputs saved to: {os.path.abspath(output_dir)}/")

## 9. Summary and Next Steps

### What We Learned:

1. **PySUS provides easy access** to Brazilian health data from DATASUS
2. **Multiple databases available**: SINAN (diseases), SIM (mortality), SIH (hospitalizations)
3. **Data includes**: demographics, dates, locations, diagnoses, and outcomes
4. **Analysis possibilities**: Time series, geographic comparisons, risk factors

### Next Steps:

- 🔍 **Explore other diseases**: Chikungunya, Zika, Malaria, COVID-19
- 🗺️ **Geospatial analysis**: Map cases by municipality
- 📈 **Time series forecasting**: Predict future outbreaks
- 🔗 **Merge datasets**: Link SINAN with SIM for outcome analysis

### Resources:

- PySUS Documentation: https://pysus.readthedocs.io/
- DATASUS: https://datasus.saude.gov.br/
- SINAN Manual: https://portalsinan.saude.gov.br/

---

**Notebook created:** 2024
**Author:** Epidemiological Datasets Repository